# Notebook 02: SageMaker Training and Deployment

**Module 3: MLOps & Production Systems | Week 11: SageMaker Training**

---

## Learning Objectives

1. Understand Amazon SageMaker's core components and when to use them
2. Set up SageMaker sessions, roles, and default buckets for training
3. Build a local SageMaker simulator to prototype training workflows without AWS costs
4. Train models using SageMaker's built-in algorithms (XGBoost)
5. Write custom training scripts using Script Mode (the most common real-world pattern)
6. Perform Hyperparameter Optimization (HPO) with budget-aware search strategies
7. Deploy models to real-time, batch, and serverless endpoints
8. Run data processing jobs at scale with SageMaker Processing
9. Estimate and manage SageMaker costs effectively
10. Know when SageMaker is overkill and simpler alternatives suffice

## Resources

- [SageMaker Developer Guide](https://docs.aws.amazon.com/sagemaker/latest/dg/whatis.html)
- [SageMaker Python SDK](https://sagemaker.readthedocs.io/en/stable/)
- [SageMaker Examples (GitHub)](https://github.com/aws/amazon-sagemaker-examples)
- [SageMaker Pricing](https://aws.amazon.com/sagemaker/pricing/)
- [XGBoost on SageMaker](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html)

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_auc_score
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.datasets import make_classification
import json
import os
import tempfile
import time
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("All imports successful.")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

---

## Section 1: SageMaker Overview

Amazon SageMaker is a fully managed machine learning platform that covers the entire ML lifecycle:
data labeling, feature engineering, training, tuning, deployment, and monitoring.

### Core Components

| Component | What it Does | When to Use |
|---|---|---|
| **Training Jobs** | Run model training on managed compute | Datasets too large for your laptop, GPU-intensive training |
| **Hyperparameter Optimization** | Automated hyperparameter search | When you need to tune 3+ hyperparameters efficiently |
| **Endpoints** | Host models for real-time inference | Production APIs needing <100ms latency |
| **Batch Transform** | Offline batch predictions | Scoring millions of records, no real-time requirement |
| **Processing Jobs** | Data preprocessing / postprocessing | ETL at scale, feature engineering, evaluation |
| **Feature Store** | Centralized feature repository | Teams sharing features, online+offline serving |
| **Pipelines** | ML workflow orchestration | Reproducible, automated training pipelines |
| **Model Monitor** | Detect data/model drift | Production models needing quality assurance |

### When to Use SageMaker vs Simpler Alternatives

| Scenario | Recommendation | Reasoning |
|---|---|---|
| Dataset < 1 GB, simple model | **Local / notebook** | Overhead of SageMaker not worth it |
| Dataset 1-10 GB, standard ML | **SageMaker or EC2** | Depends on team infrastructure |
| Dataset > 10 GB, distributed training | **SageMaker** | Managed distributed training is a huge win |
| GPU training (deep learning) | **SageMaker** | Spot instances save 60-90% on GPU costs |
| Quick prototype / experiment | **Local** | Iterate fast, deploy later |
| Production deployment with auto-scaling | **SageMaker Endpoints** | Managed scaling, monitoring, A/B testing |
| Batch scoring once a day | **SageMaker Batch Transform** or **AWS Lambda** | Depends on data size |

### Pricing Model

SageMaker uses **pay-per-second** billing for training instances:
- You pay only while your training job is running
- No upfront commitment required
- Spot training can reduce costs by up to 90%
- Endpoints are billed per-second while deployed (even if idle!)

### Architecture Diagram

```
+-----------------+      +-------------------+      +------------------+
|   S3 Bucket     |      |  SageMaker        |      |  SageMaker       |
|  (Training Data)|----->|  Training Job      |----->|  Model Artifact  |
|                 |      |  (EC2 Instance)    |      |  (S3)            |
+-----------------+      +-------------------+      +------------------+
                                                            |
                                                            v
+-----------------+      +-------------------+      +------------------+
|   Client App    |<-----|  SageMaker        |<-----|  Model Endpoint  |
|   (API call)    |      |  Endpoint         |      |  Configuration   |
+-----------------+      +-------------------+      +------------------+

Key Data Flow:
1. Upload training data to S3
2. Launch Training Job (SageMaker provisions compute, pulls data, trains)
3. Model artifact saved to S3
4. Create endpoint from model artifact
5. Client sends prediction requests to endpoint
```

In [ ]:
# Visualize SageMaker workflow
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

steps = [
    ('1. Data\nPreparation', 'Upload to S3\nor Feature Store'),
    ('2. Training\nJob', 'Built-in algo\nor Script Mode'),
    ('3. HPO\n(Optional)', 'Bayesian or\nRandom search'),
    ('4. Model\nRegistry', 'Version and\napprove models'),
    ('5. Deploy\nEndpoint', 'Real-time,\nBatch, or Serverless'),
    ('6. Monitor', 'Data drift,\nmodel quality')
]

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#00BCD4']

for i, ((title, desc), color) in enumerate(zip(steps, colors)):
    x = i * 2.2
    rect = plt.Rectangle((x, 1), 1.8, 2.5, linewidth=2, edgecolor=color,
                          facecolor=color, alpha=0.15, zorder=2)
    ax.add_patch(rect)
    ax.text(x + 0.9, 2.8, title, ha='center', va='center', fontsize=10, fontweight='bold')
    ax.text(x + 0.9, 1.7, desc, ha='center', va='center', fontsize=8, style='italic')
    if i < len(steps) - 1:
        ax.annotate('', xy=(x + 2.0, 2.25), xytext=(x + 1.85, 2.25),
                     arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax.set_xlim(-0.5, 13)
ax.set_ylim(0, 4.5)
ax.set_title('SageMaker ML Workflow', fontsize=14, fontweight='bold', pad=15)
ax.axis('off')
plt.tight_layout()
plt.show()

---

## Section 2: SageMaker Session and Role Setup

Before any SageMaker operation, you need three things:

1. **Session** (`sagemaker.Session()`): Manages interactions with SageMaker APIs and S3.
   It holds your default S3 bucket, region, and boto3 session.

2. **Role** (`sagemaker.get_execution_role()`): The IAM role that SageMaker assumes to
   access S3, ECR, CloudWatch, etc. In a notebook instance this is auto-detected;
   locally you must specify it as an ARN string.

3. **Default Bucket**: SageMaker creates `sagemaker-{region}-{account_id}` automatically.
   Training data, model artifacts, and outputs are stored here by default.

### Local Development Pattern

For local development and testing (which is what we do in this notebook), we simulate
these components so you can learn the SageMaker API patterns without incurring AWS costs.

We use a `USE_REAL_AWS = False` flag. When set to `True`, the notebook will attempt
to use real SageMaker resources.

In [ ]:
# Configuration flag: set to True if you have AWS credentials and want to use real SageMaker
USE_REAL_AWS = False

# Attempt to import SageMaker SDK
SAGEMAKER_AVAILABLE = False
try:
    import sagemaker
    from sagemaker import get_execution_role
    from sagemaker.estimator import Estimator
    SAGEMAKER_AVAILABLE = True
    print(f"SageMaker SDK version: {sagemaker.__version__}")
except ImportError:
    print("SageMaker SDK not installed. Using local simulation mode.")
    print("To install: pip install sagemaker")

if USE_REAL_AWS and SAGEMAKER_AVAILABLE:
    try:
        session = sagemaker.Session()
        role = get_execution_role()
        bucket = session.default_bucket()
        region = session.boto_region_name
        print(f"SageMaker session created.")
        print(f"  Role: {role[:50]}...")
        print(f"  Bucket: {bucket}")
        print(f"  Region: {region}")
    except Exception as e:
        print(f"Could not create SageMaker session: {e}")
        print("Falling back to simulation mode.")
        USE_REAL_AWS = False
else:
    USE_REAL_AWS = False
    print("Running in LOCAL SIMULATION mode.")
    print("All SageMaker operations will be simulated using sklearn.")
    print("Set USE_REAL_AWS = True and install sagemaker SDK to use real AWS.")

In [ ]:
# Generate a synthetic churn dataset that we'll use throughout this notebook
np.random.seed(42)

n_samples = 2000

churn_data = pd.DataFrame({
    'tenure_months': np.random.randint(1, 72, n_samples),
    'monthly_charges': np.random.uniform(20, 120, n_samples),
    'total_charges': np.random.uniform(100, 8000, n_samples),
    'num_support_tickets': np.random.poisson(2, n_samples),
    'num_referrals': np.random.poisson(1, n_samples),
    'contract_type': np.random.choice([0, 1, 2], n_samples, p=[0.5, 0.3, 0.2]),  # month-to-month, 1yr, 2yr
    'online_security': np.random.choice([0, 1], n_samples, p=[0.4, 0.6]),
    'tech_support': np.random.choice([0, 1], n_samples, p=[0.5, 0.5]),
    'paperless_billing': np.random.choice([0, 1], n_samples, p=[0.4, 0.6]),
    'payment_method': np.random.choice([0, 1, 2, 3], n_samples),
})

# Create a realistic churn target based on features
churn_prob = (
    0.3
    - 0.004 * churn_data['tenure_months']
    + 0.003 * churn_data['monthly_charges']
    + 0.02 * churn_data['num_support_tickets']
    - 0.15 * churn_data['contract_type']
    - 0.05 * churn_data['online_security']
    - 0.05 * churn_data['tech_support']
    + 0.05 * churn_data['paperless_billing']
    + np.random.normal(0, 0.1, n_samples)
)
churn_prob = np.clip(churn_prob, 0.05, 0.95)
churn_data['churn'] = (np.random.random(n_samples) < churn_prob).astype(int)

print(f"Dataset shape: {churn_data.shape}")
print(f"Churn rate: {churn_data['churn'].mean():.2%}")
print(f"\nFeature summary:")
churn_data.describe().round(2)

In [ ]:
# Prepare train/validation/test splits and save to local "S3-like" paths
features = churn_data.drop('churn', axis=1)
target = churn_data['churn']

X_train, X_temp, y_train, y_temp = train_test_split(features, target, test_size=0.3, random_state=42, stratify=target)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Training set:   {X_train.shape[0]} samples ({y_train.mean():.2%} churn)")
print(f"Validation set: {X_val.shape[0]} samples ({y_val.mean():.2%} churn)")
print(f"Test set:       {X_test.shape[0]} samples ({y_test.mean():.2%} churn)")

# Save to temp directory (simulating S3 channels)
DATA_DIR = tempfile.mkdtemp(prefix='sagemaker_sim_')
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR = os.path.join(DATA_DIR, 'validation')
TEST_DIR = os.path.join(DATA_DIR, 'test')
MODEL_DIR = os.path.join(DATA_DIR, 'model')
OUTPUT_DIR = os.path.join(DATA_DIR, 'output')

for d in [TRAIN_DIR, VAL_DIR, TEST_DIR, MODEL_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# SageMaker convention: target column first, no header, CSV format
train_df = pd.concat([y_train.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
val_df = pd.concat([y_val.reset_index(drop=True), X_val.reset_index(drop=True)], axis=1)
test_df = pd.concat([y_test.reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)

train_df.to_csv(os.path.join(TRAIN_DIR, 'train.csv'), index=False, header=False)
val_df.to_csv(os.path.join(VAL_DIR, 'validation.csv'), index=False, header=False)
test_df.to_csv(os.path.join(TEST_DIR, 'test.csv'), index=False, header=False)

print(f"\nData saved to: {DATA_DIR}")
print(f"  Train: {os.path.join(TRAIN_DIR, 'train.csv')}")
print(f"  Val:   {os.path.join(VAL_DIR, 'validation.csv')}")
print(f"  Test:  {os.path.join(TEST_DIR, 'test.csv')}")

---

## Section 3: SageMakerSimulator Class

Since we want to learn SageMaker's API patterns without incurring AWS costs, we build a
**SageMakerSimulator** that wraps sklearn in a SageMaker-like interface.

This gives us the best of both worlds:
- Learn the exact SageMaker API (Estimator, fit, deploy, predict)
- Run everything locally and instantly
- When ready for production, swap `SageMakerSimulator` for real `sagemaker` calls

### Key Design Decisions

1. **Estimator.fit()** reads CSV data from local directories (simulating S3 input channels)
2. **Estimator.deploy()** returns a MockEndpoint with predict/delete methods
3. We support `xgboost` (via GradientBoosting) and `sklearn` algorithm types
4. Hyperparameters map directly to sklearn constructor arguments
5. Training logs mimic real SageMaker CloudWatch output format

In [ ]:
import pickle
from datetime import datetime


class SageMakerSimulator:
    """Simulates SageMaker training locally using sklearn.
    
    Provides a SageMaker-compatible API so you can learn the patterns
    without incurring AWS costs. When ready for production, replace
    SageMakerSimulator calls with real sagemaker SDK calls.
    """
    
    # Simulated container URIs (in real SageMaker these point to ECR)
    ALGORITHM_CONTAINERS = {
        'xgboost': '683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1',
        'linear-learner': '382416733822.dkr.ecr.us-east-1.amazonaws.com/linear-learner:latest',
        'sklearn': '683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1',
    }
    
    class Estimator:
        """Simulates sagemaker.estimator.Estimator.
        
        Parameters match the real SageMaker Estimator constructor.
        """
        
        def __init__(self, algorithm='xgboost', hyperparameters=None,
                     instance_type='ml.m5.xlarge', instance_count=1,
                     output_path=None, role='mock-role',
                     image_uri=None, framework_version=None,
                     base_job_name='simulator'):
            self.algorithm = algorithm
            self.hyperparameters = hyperparameters or {}
            self.instance_type = instance_type
            self.instance_count = instance_count
            self.output_path = output_path or tempfile.mkdtemp()
            self.role = role
            self.image_uri = image_uri
            self.base_job_name = base_job_name
            self.model_ = None
            self.training_job_name = None
            self.training_time_ = None
            self.metrics_ = {}
            self.feature_names_ = None
            
            print(f"[SageMaker Sim] Estimator created:")
            print(f"  Algorithm: {algorithm}")
            print(f"  Instance: {instance_type} x {instance_count}")
            print(f"  Output: {self.output_path}")
        
        def set_hyperparameters(self, **kwargs):
            """Set hyperparameters (matches SageMaker API)."""
            self.hyperparameters.update(kwargs)
            print(f"[SageMaker Sim] Hyperparameters set: {self.hyperparameters}")
        
        def _build_model(self):
            """Build the appropriate sklearn model based on algorithm type."""
            hp = self.hyperparameters
            
            if self.algorithm == 'xgboost':
                return GradientBoostingClassifier(
                    n_estimators=int(hp.get('num_round', hp.get('n_estimators', 100))),
                    max_depth=int(hp.get('max_depth', 5)),
                    learning_rate=float(hp.get('eta', hp.get('learning_rate', 0.1))),
                    subsample=float(hp.get('subsample', 0.8)),
                    min_samples_split=int(hp.get('min_child_weight', 2)),
                    random_state=42
                )
            elif self.algorithm == 'random_forest':
                return RandomForestClassifier(
                    n_estimators=int(hp.get('n_estimators', 100)),
                    max_depth=int(hp.get('max_depth', 10)),
                    min_samples_split=int(hp.get('min_samples_split', 2)),
                    random_state=42
                )
            elif self.algorithm == 'logistic':
                return LogisticRegression(
                    C=float(hp.get('C', 1.0)),
                    max_iter=int(hp.get('max_iter', 1000)),
                    random_state=42
                )
            else:
                # Default to GradientBoosting
                return GradientBoostingClassifier(
                    n_estimators=100, max_depth=5, random_state=42
                )
        
        def fit(self, inputs, wait=True, logs=True):
            """Train the model (simulates SageMaker training job).
            
            Args:
                inputs: Dict mapping channel names to local paths.
                        e.g., {'train': '/path/to/train', 'validation': '/path/to/val'}
                wait: Whether to wait for completion (always True in simulation).
                logs: Whether to print training logs.
            """
            self.training_job_name = f"{self.base_job_name}-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
            
            if logs:
                print(f"\n{'='*60}")
                print(f"[SageMaker Sim] Training job: {self.training_job_name}")
                print(f"{'='*60}")
                print(f"Instance type: {self.instance_type}")
                print(f"Instance count: {self.instance_count}")
                print(f"Hyperparameters: {json.dumps(self.hyperparameters, indent=2)}")
                print(f"Input channels: {list(inputs.keys())}")
                print(f"{'='*60}")
            
            start_time = time.time()
            
            # Load training data
            train_path = inputs.get('train', '')
            train_files = [f for f in os.listdir(train_path) if f.endswith('.csv')]
            if not train_files:
                raise ValueError(f"No CSV files found in {train_path}")
            
            train_df = pd.read_csv(os.path.join(train_path, train_files[0]), header=None)
            y_train = train_df.iloc[:, 0]
            X_train = train_df.iloc[:, 1:]
            self.feature_names_ = [f'feature_{i}' for i in range(X_train.shape[1])]
            
            if logs:
                print(f"[Training] Loaded {len(train_df)} training samples")
                print(f"[Training] Features: {X_train.shape[1]}")
                print(f"[Training] Target distribution: {dict(y_train.value_counts())}")
            
            # Load validation data if provided
            X_val, y_val = None, None
            if 'validation' in inputs:
                val_path = inputs['validation']
                val_files = [f for f in os.listdir(val_path) if f.endswith('.csv')]
                if val_files:
                    val_df = pd.read_csv(os.path.join(val_path, val_files[0]), header=None)
                    y_val = val_df.iloc[:, 0]
                    X_val = val_df.iloc[:, 1:]
                    if logs:
                        print(f"[Training] Loaded {len(val_df)} validation samples")
            
            # Build and train model
            self.model_ = self._build_model()
            
            if logs:
                print(f"\n[Training] Starting training...")
            
            self.model_.fit(X_train, y_train)
            
            # Calculate metrics
            train_pred = self.model_.predict(X_train)
            self.metrics_['train:accuracy'] = accuracy_score(y_train, train_pred)
            self.metrics_['train:f1'] = f1_score(y_train, train_pred, average='weighted')
            
            if X_val is not None:
                val_pred = self.model_.predict(X_val)
                self.metrics_['validation:accuracy'] = accuracy_score(y_val, val_pred)
                self.metrics_['validation:f1'] = f1_score(y_val, val_pred, average='weighted')
                try:
                    val_proba = self.model_.predict_proba(X_val)[:, 1]
                    self.metrics_['validation:auc'] = roc_auc_score(y_val, val_proba)
                except Exception:
                    pass
            
            self.training_time_ = time.time() - start_time
            
            # Save model artifact
            model_path = os.path.join(self.output_path, 'model.pkl')
            with open(model_path, 'wb') as f:
                pickle.dump(self.model_, f)
            
            if logs:
                print(f"\n[Training] Training complete!")
                print(f"[Training] Duration: {self.training_time_:.2f}s")
                print(f"[Training] Metrics:")
                for k, v in self.metrics_.items():
                    print(f"  {k}: {v:.4f}")
                print(f"[Training] Model saved to: {model_path}")
                print(f"{'='*60}\n")
            
            return self
        
        def deploy(self, instance_type='ml.m5.large', initial_instance_count=1,
                   endpoint_name=None):
            """Deploy model to a mock endpoint."""
            if self.model_ is None:
                raise ValueError("Model not trained yet. Call fit() first.")
            
            endpoint_name = endpoint_name or f"{self.base_job_name}-endpoint"
            print(f"[SageMaker Sim] Deploying model to endpoint: {endpoint_name}")
            print(f"  Instance type: {instance_type}")
            print(f"  Instance count: {initial_instance_count}")
            
            return SageMakerSimulator.MockEndpoint(
                model=self.model_,
                endpoint_name=endpoint_name,
                instance_type=instance_type
            )
        
        def get_feature_importance(self):
            """Return feature importance if available."""
            if hasattr(self.model_, 'feature_importances_'):
                return dict(zip(self.feature_names_, self.model_.feature_importances_))
            return None
    
    class MockEndpoint:
        """Simulates a SageMaker real-time endpoint."""
        
        def __init__(self, model, endpoint_name, instance_type):
            self.model = model
            self.endpoint_name = endpoint_name
            self.instance_type = instance_type
            self.is_active = True
            self.invocation_count = 0
            print(f"[Endpoint] {endpoint_name} is InService.")
        
        def predict(self, data):
            """Make predictions (simulates InvokeEndpoint).
            
            Args:
                data: numpy array, pandas DataFrame, or list of lists.
            
            Returns:
                dict with 'predictions' and 'probabilities' keys.
            """
            if not self.is_active:
                raise RuntimeError(f"Endpoint {self.endpoint_name} has been deleted.")
            
            self.invocation_count += 1
            
            if isinstance(data, list):
                data = np.array(data)
            elif isinstance(data, pd.DataFrame):
                data = data.values
            
            if data.ndim == 1:
                data = data.reshape(1, -1)
            
            predictions = self.model.predict(data)
            
            result = {'predictions': predictions.tolist()}
            
            if hasattr(self.model, 'predict_proba'):
                probabilities = self.model.predict_proba(data)
                result['probabilities'] = probabilities.tolist()
            
            return result
        
        def delete_endpoint(self):
            """Delete the endpoint (stop billing)."""
            self.is_active = False
            print(f"[Endpoint] {self.endpoint_name} deleted.")
            print(f"  Total invocations: {self.invocation_count}")
    
    class HyperparameterTuner:
        """Simulates SageMaker HyperparameterTuner using sklearn RandomizedSearchCV."""
        
        def __init__(self, estimator, hyperparameter_ranges, objective_metric_name,
                     objective_type='Maximize', max_jobs=10, max_parallel_jobs=2,
                     strategy='Bayesian'):
            self.estimator = estimator
            self.hyperparameter_ranges = hyperparameter_ranges
            self.objective_metric_name = objective_metric_name
            self.objective_type = objective_type
            self.max_jobs = max_jobs
            self.max_parallel_jobs = max_parallel_jobs
            self.strategy = strategy
            self.results_ = []
            self.best_estimator_ = None
            self.best_params_ = None
            self.best_score_ = None
        
        def fit(self, inputs):
            """Run hyperparameter optimization."""
            print(f"\n{'='*60}")
            print(f"[SageMaker Sim] HPO Job Starting")
            print(f"{'='*60}")
            print(f"Strategy: {self.strategy}")
            print(f"Max jobs: {self.max_jobs}")
            print(f"Objective: {self.objective_type} {self.objective_metric_name}")
            print(f"Hyperparameter ranges: {json.dumps({k: str(v) for k, v in self.hyperparameter_ranges.items()}, indent=2)}")
            
            # Load data
            train_path = inputs.get('train', '')
            train_files = [f for f in os.listdir(train_path) if f.endswith('.csv')]
            train_df = pd.read_csv(os.path.join(train_path, train_files[0]), header=None)
            y_train = train_df.iloc[:, 0]
            X_train = train_df.iloc[:, 1:]
            
            X_val, y_val = None, None
            if 'validation' in inputs:
                val_path = inputs['validation']
                val_files = [f for f in os.listdir(val_path) if f.endswith('.csv')]
                if val_files:
                    val_df = pd.read_csv(os.path.join(val_path, val_files[0]), header=None)
                    y_val = val_df.iloc[:, 0]
                    X_val = val_df.iloc[:, 1:]
            
            # Run HPO trials
            best_score = -np.inf if self.objective_type == 'Maximize' else np.inf
            
            for trial in range(self.max_jobs):
                # Sample hyperparameters
                sampled_hp = {}
                for name, values in self.hyperparameter_ranges.items():
                    if isinstance(values, list):
                        sampled_hp[name] = np.random.choice(values)
                    elif isinstance(values, tuple) and len(values) == 2:
                        low, high = values
                        if isinstance(low, int) and isinstance(high, int):
                            sampled_hp[name] = np.random.randint(low, high + 1)
                        else:
                            sampled_hp[name] = np.random.uniform(low, high)
                    else:
                        sampled_hp[name] = values
                
                # Create and train model
                trial_estimator = SageMakerSimulator.Estimator(
                    algorithm=self.estimator.algorithm,
                    hyperparameters=sampled_hp,
                    instance_type=self.estimator.instance_type,
                    output_path=self.estimator.output_path,
                    base_job_name=f"{self.estimator.base_job_name}-trial-{trial}"
                )
                trial_estimator.fit(inputs, logs=False)
                
                # Get metric
                metric_key = f'validation:{self.objective_metric_name}'
                score = trial_estimator.metrics_.get(metric_key, 0)
                
                result = {
                    'trial': trial + 1,
                    'hyperparameters': sampled_hp.copy(),
                    'score': score,
                    'status': 'Completed'
                }
                self.results_.append(result)
                
                is_better = (score > best_score if self.objective_type == 'Maximize' 
                             else score < best_score)
                if is_better:
                    best_score = score
                    self.best_estimator_ = trial_estimator
                    self.best_params_ = sampled_hp.copy()
                    self.best_score_ = score
                
                print(f"  Trial {trial + 1}/{self.max_jobs}: score={score:.4f} | params={sampled_hp}")
            
            print(f"\n[HPO] Best score: {self.best_score_:.4f}")
            print(f"[HPO] Best params: {self.best_params_}")
            print(f"{'='*60}\n")
            
            return self
        
        def results_as_dataframe(self):
            """Return HPO results as a DataFrame."""
            rows = []
            for r in self.results_:
                row = {'trial': r['trial'], 'score': r['score'], 'status': r['status']}
                row.update(r['hyperparameters'])
                rows.append(row)
            return pd.DataFrame(rows)


print("SageMakerSimulator class defined successfully.")
print("Available components:")
print("  - SageMakerSimulator.Estimator")
print("  - SageMakerSimulator.MockEndpoint")
print("  - SageMakerSimulator.HyperparameterTuner")

In [ ]:
# Quick test of the simulator
test_est = SageMakerSimulator.Estimator(
    algorithm='xgboost',
    hyperparameters={'max_depth': 3, 'num_round': 50, 'eta': 0.1},
    instance_type='ml.m5.xlarge',
    output_path=MODEL_DIR,
    base_job_name='quick-test'
)

# Verify it can access our data
print(f"\nTrain dir contents: {os.listdir(TRAIN_DIR)}")
print(f"Val dir contents: {os.listdir(VAL_DIR)}")

---

## Section 4: Training with Built-in Algorithms

SageMaker provides several **built-in algorithms** that are optimized for distributed training:

| Algorithm | Use Case | Key Advantage |
|---|---|---|
| **XGBoost** | Classification, regression | Most popular; highly optimized |
| **Linear Learner** | Linear/logistic regression | Fast, distributed training |
| **KNN** | Classification, regression | Simple, no training phase |
| **Factorization Machines** | Recommendations | Handles sparse data |
| **BlazingText** | Text classification, Word2Vec | Very fast NLP |
| **Image Classification** | Computer vision | Transfer learning built-in |

### XGBoost on SageMaker

XGBoost is by far the most used built-in algorithm. The SageMaker version supports:
- Distributed training across multiple instances
- GPU training (set `tree_method='gpu_hist'`)
- Automatic model tuning (HPO)
- CSV and libsvm input formats

**Important**: SageMaker's XGBoost expects the target column to be the **first column** in CSV, with **no header row**.

### Real SageMaker Code (for reference)

```python
# --- REAL SAGEMAKER CODE (do not run in simulation mode) ---
import sagemaker
from sagemaker import image_uris

# Get the XGBoost container URI for your region
container = image_uris.retrieve('xgboost', sagemaker.Session().boto_region_name, '1.7-1')

# Create estimator
xgb = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{bucket}/xgboost-output',
    sagemaker_session=session
)

xgb.set_hyperparameters(
    max_depth=5,
    eta=0.2,
    num_round=100,
    objective='binary:logistic',
    eval_metric='auc'
)

# Specify S3 input channels
from sagemaker.inputs import TrainingInput
train_input = TrainingInput(s3_data=f's3://{bucket}/data/train', content_type='csv')
val_input = TrainingInput(s3_data=f's3://{bucket}/data/validation', content_type='csv')

xgb.fit({'train': train_input, 'validation': val_input})
```

In [ ]:
# Train XGBoost using our SageMaker Simulator
xgb_estimator = SageMakerSimulator.Estimator(
    algorithm='xgboost',
    hyperparameters={
        'max_depth': 5,
        'eta': 0.2,
        'num_round': 100,
        'subsample': 0.8,
        'objective': 'binary:logistic',  # stored but sklearn uses its own
        'eval_metric': 'auc',
    },
    instance_type='ml.m5.xlarge',
    instance_count=1,
    output_path=MODEL_DIR,
    base_job_name='churn-xgboost'
)

# Train with train and validation channels (just like real SageMaker)
xgb_estimator.fit({
    'train': TRAIN_DIR,
    'validation': VAL_DIR
})

In [ ]:
# Print detailed training metrics
print("Training Metrics Summary")
print("=" * 40)
for metric_name, metric_value in xgb_estimator.metrics_.items():
    print(f"  {metric_name:30s}: {metric_value:.4f}")

# Feature importance
importance = xgb_estimator.get_feature_importance()
if importance:
    feature_map = {
        'feature_0': 'tenure_months',
        'feature_1': 'monthly_charges',
        'feature_2': 'total_charges',
        'feature_3': 'num_support_tickets',
        'feature_4': 'num_referrals',
        'feature_5': 'contract_type',
        'feature_6': 'online_security',
        'feature_7': 'tech_support',
        'feature_8': 'paperless_billing',
        'feature_9': 'payment_method',
    }
    importance_named = {feature_map.get(k, k): v for k, v in importance.items()}
    importance_df = pd.DataFrame(
        sorted(importance_named.items(), key=lambda x: x[1], reverse=True),
        columns=['Feature', 'Importance']
    )
    print(f"\nFeature Importance:")
    print(importance_df.to_string(index=False))

In [ ]:
# Visualize feature importance
if importance:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart of feature importance
    importance_df_sorted = importance_df.sort_values('Importance', ascending=True)
    axes[0].barh(importance_df_sorted['Feature'], importance_df_sorted['Importance'],
                 color=sns.color_palette('viridis', len(importance_df_sorted)))
    axes[0].set_xlabel('Importance')
    axes[0].set_title('XGBoost Feature Importance (Churn Prediction)')
    
    # Confusion matrix on validation set
    val_df = pd.read_csv(os.path.join(VAL_DIR, 'validation.csv'), header=None)
    y_val_actual = val_df.iloc[:, 0]
    X_val_data = val_df.iloc[:, 1:]
    y_val_pred = xgb_estimator.model_.predict(X_val_data)
    
    cm = confusion_matrix(y_val_actual, y_val_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')
    axes[1].set_title('Validation Confusion Matrix')
    
    plt.tight_layout()
    plt.show()
    
    # Print classification report
    print("\nClassification Report (Validation Set):")
    print(classification_report(y_val_actual, y_val_pred, target_names=['No Churn', 'Churn']))

---

## Section 5: Custom Training Scripts (Script Mode)

**Script Mode is the most important SageMaker training pattern** for real-world ML engineering.

Instead of using built-in algorithms, you write your own `train.py` script and SageMaker:
1. Provisions the instance(s)
2. Pulls your training container (e.g., PyTorch, TensorFlow, sklearn)
3. Copies your script into the container
4. Sets environment variables for data channels and model output
5. Runs your script
6. Saves model artifacts to S3

### SageMaker Container Environment Variables

Your training script reads these environment variables:

| Variable | Description | Example |
|---|---|---|
| `SM_CHANNEL_TRAIN` | Path to training data | `/opt/ml/input/data/train` |
| `SM_CHANNEL_VALIDATION` | Path to validation data | `/opt/ml/input/data/validation` |
| `SM_MODEL_DIR` | Where to save the trained model | `/opt/ml/model` |
| `SM_OUTPUT_DATA_DIR` | Where to save other outputs | `/opt/ml/output/data` |
| `SM_NUM_GPUS` | Number of GPUs available | `1` |
| `SM_HP_*` | Hyperparameters (prefixed) | `SM_HP_LEARNING_RATE=0.01` |

### Container Directory Structure

```
/opt/ml/
├── input/
│   ├── config/
│   │   ├── hyperparameters.json    # Your hyperparameters
│   │   └── resourceconfig.json     # Instance info (for distributed)
│   └── data/
│       ├── train/                  # SM_CHANNEL_TRAIN
│       │   └── train.csv
│       └── validation/             # SM_CHANNEL_VALIDATION
│           └── validation.csv
├── model/                          # SM_MODEL_DIR - save model here
│   └── model.pkl
└── output/
    ├── data/                       # SM_OUTPUT_DATA_DIR
    └── failure                     # Error message if training fails
```

### Why Script Mode?

- **Flexibility**: Use any Python library, any model architecture
- **Portability**: Your script runs locally and on SageMaker identically
- **Debuggability**: Test locally first, then scale to cloud
- **Team-friendly**: Version control your training scripts

In [ ]:
# Write a SageMaker-compatible training script
# This script follows the exact patterns used in real SageMaker Script Mode

TRAIN_SCRIPT = '''
#!/usr/bin/env python3
"""
SageMaker-compatible training script for churn prediction.

This script follows the SageMaker Script Mode conventions:
- Reads data from SM_CHANNEL_TRAIN and SM_CHANNEL_VALIDATION
- Reads hyperparameters from SM_HP_* environment variables
- Saves model to SM_MODEL_DIR
- Can run locally (with env vars set) or on SageMaker
"""

import argparse
import os
import json
import pickle
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report


def parse_args():
    """Parse hyperparameters from command line or environment variables."""
    parser = argparse.ArgumentParser()
    
    # Hyperparameters
    parser.add_argument('--n-estimators', type=int, default=int(os.environ.get('SM_HP_N_ESTIMATORS', 100)))
    parser.add_argument('--max-depth', type=int, default=int(os.environ.get('SM_HP_MAX_DEPTH', 5)))
    parser.add_argument('--learning-rate', type=float, default=float(os.environ.get('SM_HP_LEARNING_RATE', 0.1)))
    parser.add_argument('--subsample', type=float, default=float(os.environ.get('SM_HP_SUBSAMPLE', 0.8)))
    
    # SageMaker environment
    parser.add_argument('--model-dir', type=str, default=os.environ.get('SM_MODEL_DIR', '/opt/ml/model'))
    parser.add_argument('--train', type=str, default=os.environ.get('SM_CHANNEL_TRAIN', '/opt/ml/input/data/train'))
    parser.add_argument('--validation', type=str, default=os.environ.get('SM_CHANNEL_VALIDATION', '/opt/ml/input/data/validation'))
    parser.add_argument('--output-data-dir', type=str, default=os.environ.get('SM_OUTPUT_DATA_DIR', '/opt/ml/output/data'))
    
    return parser.parse_args()


def load_data(channel_path):
    """Load CSV data from a SageMaker channel directory."""
    csv_files = [f for f in os.listdir(channel_path) if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError(f"No CSV files in {channel_path}")
    
    dfs = [pd.read_csv(os.path.join(channel_path, f), header=None) for f in csv_files]
    data = pd.concat(dfs, ignore_index=True)
    
    y = data.iloc[:, 0]     # Target is first column (SageMaker convention)
    X = data.iloc[:, 1:]    # Features are remaining columns
    return X, y


def train(args):
    """Main training function."""
    print("=" * 50)
    print("Starting training...")
    print(f"  n_estimators:  {args.n_estimators}")
    print(f"  max_depth:     {args.max_depth}")
    print(f"  learning_rate: {args.learning_rate}")
    print(f"  subsample:     {args.subsample}")
    print("=" * 50)
    
    # Load data
    print(f"Loading training data from {args.train}")
    X_train, y_train = load_data(args.train)
    print(f"  Training samples: {len(X_train)}")
    
    X_val, y_val = None, None
    if os.path.exists(args.validation):
        print(f"Loading validation data from {args.validation}")
        X_val, y_val = load_data(args.validation)
        print(f"  Validation samples: {len(X_val)}")
    
    # Build model
    model = GradientBoostingClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        learning_rate=args.learning_rate,
        subsample=args.subsample,
        random_state=42
    )
    
    # Train
    model.fit(X_train, y_train)
    
    # Evaluate
    train_pred = model.predict(X_train)
    train_acc = accuracy_score(y_train, train_pred)
    train_f1 = f1_score(y_train, train_pred, average='weighted')
    print(f"\nTraining metrics:")
    print(f"  accuracy: {train_acc:.4f}")
    print(f"  f1:       {train_f1:.4f}")
    
    if X_val is not None:
        val_pred = model.predict(X_val)
        val_proba = model.predict_proba(X_val)[:, 1]
        val_acc = accuracy_score(y_val, val_pred)
        val_f1 = f1_score(y_val, val_pred, average='weighted')
        val_auc = roc_auc_score(y_val, val_proba)
        print(f"\nValidation metrics:")
        print(f"  accuracy: {val_acc:.4f}")
        print(f"  f1:       {val_f1:.4f}")
        print(f"  auc:      {val_auc:.4f}")
        print(f"\nClassification Report:")
        print(classification_report(y_val, val_pred))
    
    # Save model
    os.makedirs(args.model_dir, exist_ok=True)
    model_path = os.path.join(args.model_dir, 'model.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f"\nModel saved to {model_path}")
    
    # Save metrics to output
    os.makedirs(args.output_data_dir, exist_ok=True)
    metrics = {
        'train_accuracy': train_acc,
        'train_f1': train_f1,
    }
    if X_val is not None:
        metrics['val_accuracy'] = val_acc
        metrics['val_f1'] = val_f1
        metrics['val_auc'] = val_auc
    
    with open(os.path.join(args.output_data_dir, 'metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=2)
    
    print("\nTraining complete!")
    return model


# SageMaker entry point
if __name__ == '__main__':
    args = parse_args()
    train(args)
'''

# Save the training script to a temp file
SCRIPT_DIR = os.path.join(DATA_DIR, 'scripts')
os.makedirs(SCRIPT_DIR, exist_ok=True)
script_path = os.path.join(SCRIPT_DIR, 'train.py')

with open(script_path, 'w') as f:
    f.write(TRAIN_SCRIPT)

print(f"Training script saved to: {script_path}")
print(f"\nScript length: {len(TRAIN_SCRIPT)} characters")
print(f"\nThis script can run:")
print(f"  1. Locally: python train.py --train ./data/train --validation ./data/val --model-dir ./model")
print(f"  2. On SageMaker: environment variables are set automatically")

In [ ]:
# Run the training script locally with environment variables set
# This is EXACTLY how you should test before deploying to SageMaker

import subprocess

# Set environment variables (simulating what SageMaker sets inside the container)
script_model_dir = os.path.join(DATA_DIR, 'script_model')
script_output_dir = os.path.join(DATA_DIR, 'script_output')
os.makedirs(script_model_dir, exist_ok=True)
os.makedirs(script_output_dir, exist_ok=True)

env = os.environ.copy()
env.update({
    'SM_CHANNEL_TRAIN': TRAIN_DIR,
    'SM_CHANNEL_VALIDATION': VAL_DIR,
    'SM_MODEL_DIR': script_model_dir,
    'SM_OUTPUT_DATA_DIR': script_output_dir,
    'SM_HP_N_ESTIMATORS': '150',
    'SM_HP_MAX_DEPTH': '6',
    'SM_HP_LEARNING_RATE': '0.15',
    'SM_HP_SUBSAMPLE': '0.85',
})

print("Running training script with SageMaker environment variables...")
print(f"  SM_CHANNEL_TRAIN = {TRAIN_DIR}")
print(f"  SM_CHANNEL_VALIDATION = {VAL_DIR}")
print(f"  SM_MODEL_DIR = {script_model_dir}")
print(f"  SM_HP_N_ESTIMATORS = 150")
print(f"  SM_HP_MAX_DEPTH = 6")
print()

result = subprocess.run(
    [sys.executable, script_path],
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(f"STDERR: {result.stderr}")
print(f"Return code: {result.returncode}")

In [ ]:
# Verify the script's outputs
print("Checking script outputs...")
print(f"\nModel directory: {os.listdir(script_model_dir)}")
print(f"Output directory: {os.listdir(script_output_dir)}")

# Load and inspect the saved metrics
metrics_path = os.path.join(script_output_dir, 'metrics.json')
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        script_metrics = json.load(f)
    print(f"\nSaved metrics:")
    for k, v in script_metrics.items():
        print(f"  {k}: {v:.4f}")

# Load the saved model and test it
model_pkl_path = os.path.join(script_model_dir, 'model.pkl')
if os.path.exists(model_pkl_path):
    with open(model_pkl_path, 'rb') as f:
        script_model = pickle.load(f)
    print(f"\nModel type: {type(script_model).__name__}")
    print(f"Model parameters: {script_model.get_params()}")

In [ ]:
# Show the equivalent real SageMaker Script Mode code
print("""
# ===== REAL SAGEMAKER SCRIPT MODE CODE =====
# (Do not run in simulation mode)

from sagemaker.sklearn import SKLearn

sklearn_estimator = SKLearn(
    entry_point='train.py',           # Your training script
    source_dir='./scripts',            # Directory containing the script
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    framework_version='1.2-1',
    hyperparameters={
        'n-estimators': 150,
        'max-depth': 6,
        'learning-rate': 0.15,
        'subsample': 0.85,
    },
    output_path=f's3://{bucket}/sklearn-output',
    base_job_name='churn-sklearn',
)

# Launch training job (takes ~5 minutes to provision + train)
sklearn_estimator.fit({
    'train': f's3://{bucket}/data/train',
    'validation': f's3://{bucket}/data/validation'
})

# Training job name for reference
print(f"Job: {sklearn_estimator.latest_training_job.name}")
""")

---

## Section 6: Hyperparameter Optimization

SageMaker Hyperparameter Optimization (HPO) automates the search for the best
hyperparameters by launching multiple training jobs with different configurations.

### Key Concepts

1. **Objective Metric**: The metric to optimize (e.g., `validation:auc`)
2. **Hyperparameter Ranges**: Continuous, Integer, or Categorical ranges for each HP
3. **Strategy**: How to explore the search space
4. **Resource Limits**: Max training jobs, max parallel jobs

### Search Strategies

| Strategy | Description | Best When |
|---|---|---|
| **Bayesian** (default) | Uses past results to guide next trial | Budget > 10 trials |
| **Random** | Samples randomly from ranges | Quick exploration, any budget |
| **Grid** | Exhaustive grid search | Small, discrete search space |
| **Hyperband** | Early stopping of bad trials | Large budgets, expensive training |

### Real SageMaker HPO Code (for reference)

```python
from sagemaker.tuner import (
    HyperparameterTuner, IntegerParameter,
    ContinuousParameter, CategoricalParameter
)

hyperparameter_ranges = {
    'max_depth': IntegerParameter(3, 10),
    'eta': ContinuousParameter(0.01, 0.3),
    'num_round': IntegerParameter(50, 300),
    'subsample': ContinuousParameter(0.5, 1.0),
    'colsample_bytree': ContinuousParameter(0.5, 1.0),
}

tuner = HyperparameterTuner(
    estimator=xgb_estimator,
    objective_metric_name='validation:auc',
    objective_type='Maximize',
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=20,
    max_parallel_jobs=4,
    strategy='Bayesian',
)

tuner.fit({'train': train_input, 'validation': val_input})
```

In [ ]:
# Define hyperparameter ranges for HPO
hyperparameter_ranges = {
    'max_depth': (3, 10),              # IntegerParameter
    'eta': (0.01, 0.3),                # ContinuousParameter
    'num_round': (50, 300),            # IntegerParameter
    'subsample': (0.5, 1.0),           # ContinuousParameter
}

# Create base estimator for HPO
base_estimator = SageMakerSimulator.Estimator(
    algorithm='xgboost',
    instance_type='ml.m5.xlarge',
    output_path=MODEL_DIR,
    base_job_name='churn-hpo'
)

# Create and run HPO
tuner = SageMakerSimulator.HyperparameterTuner(
    estimator=base_estimator,
    hyperparameter_ranges=hyperparameter_ranges,
    objective_metric_name='accuracy',
    objective_type='Maximize',
    max_jobs=15,
    max_parallel_jobs=3,
    strategy='Bayesian'
)

tuner.fit({
    'train': TRAIN_DIR,
    'validation': VAL_DIR
})

In [ ]:
# Display HPO results as a table
results_df = tuner.results_as_dataframe()
results_df = results_df.sort_values('score', ascending=False)

print("HPO Results (sorted by score):")
print("=" * 80)
print(results_df.to_string(index=False, float_format='%.4f'))
print(f"\nBest trial: #{results_df.iloc[0]['trial']:.0f}")
print(f"Best score: {results_df.iloc[0]['score']:.4f}")

In [ ]:
# Visualize HPO results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Score progression
ax = axes[0, 0]
trial_order_df = tuner.results_as_dataframe().sort_values('trial')
ax.plot(trial_order_df['trial'], trial_order_df['score'], 'bo-', alpha=0.6, label='Trial score')
ax.axhline(y=tuner.best_score_, color='r', linestyle='--', label=f'Best: {tuner.best_score_:.4f}')
cummax = trial_order_df['score'].cummax()
ax.plot(trial_order_df['trial'], cummax, 'g-', linewidth=2, label='Best so far')
ax.set_xlabel('Trial')
ax.set_ylabel('Validation Accuracy')
ax.set_title('HPO Score Progression')
ax.legend()

# Score vs max_depth
ax = axes[0, 1]
ax.scatter(results_df['max_depth'], results_df['score'], c=results_df['score'],
           cmap='viridis', s=80, edgecolors='black', alpha=0.7)
ax.set_xlabel('max_depth')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Score vs max_depth')

# Score vs learning rate
ax = axes[1, 0]
ax.scatter(results_df['eta'], results_df['score'], c=results_df['score'],
           cmap='viridis', s=80, edgecolors='black', alpha=0.7)
ax.set_xlabel('eta (learning rate)')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Score vs Learning Rate')

# Score vs num_round
ax = axes[1, 1]
ax.scatter(results_df['num_round'], results_df['score'], c=results_df['score'],
           cmap='viridis', s=80, edgecolors='black', alpha=0.7)
ax.set_xlabel('num_round')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Score vs num_round')

plt.suptitle('Hyperparameter Optimization Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate the best model from HPO on the test set
print("Evaluating best HPO model on test set...")
print(f"Best hyperparameters: {tuner.best_params_}")

test_df = pd.read_csv(os.path.join(TEST_DIR, 'test.csv'), header=None)
y_test_actual = test_df.iloc[:, 0]
X_test_data = test_df.iloc[:, 1:]

best_model = tuner.best_estimator_.model_
y_test_pred = best_model.predict(X_test_data)
y_test_proba = best_model.predict_proba(X_test_data)[:, 1]

test_acc = accuracy_score(y_test_actual, y_test_pred)
test_f1 = f1_score(y_test_actual, y_test_pred, average='weighted')
test_auc = roc_auc_score(y_test_actual, y_test_proba)

print(f"\nTest Set Results:")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  F1 Score: {test_f1:.4f}")
print(f"  AUC:      {test_auc:.4f}")
print(f"\n{classification_report(y_test_actual, y_test_pred, target_names=['No Churn', 'Churn'])}")

---

## Section 7: Model Deployment — Endpoints

SageMaker offers three deployment options:

### Deployment Types

| Type | Latency | Cost Model | Best For |
|---|---|---|---|
| **Real-time Endpoint** | <100ms | Pay per second (always on) | Production APIs, low latency |
| **Batch Transform** | Minutes | Pay per second (job duration) | Scoring millions of records |
| **Serverless Endpoint** | Cold start ~1s | Pay per invocation | Infrequent traffic, cost-sensitive |
| **Async Inference** | Seconds-minutes | Pay per second | Large payloads, queued processing |

### Endpoint Configuration

For real-time endpoints:
- **Instance type**: Choose based on model size and latency requirements
- **Auto-scaling**: Scale instance count based on `InvocationsPerInstance` or `CPUUtilization`
- **Multi-model endpoints**: Host multiple models on one instance to reduce cost
- **A/B testing**: Route traffic between model variants

### Request/Response Format

SageMaker endpoints accept and return data in configurable formats:
- **CSV**: `ContentType: text/csv`
- **JSON**: `ContentType: application/json`
- **JSON Lines**: `ContentType: application/jsonlines`

Example JSON request:
```json
{
    "instances": [
        {"features": [24, 79.99, 1920.50, 3, 1, 0, 1, 0, 1, 2]}
    ]
}
```

Example JSON response:
```json
{
    "predictions": [1],
    "probabilities": [[0.15, 0.85]]
}
```

In [ ]:
# Deploy the best model from HPO to a mock endpoint
endpoint = tuner.best_estimator_.deploy(
    instance_type='ml.m5.large',
    initial_instance_count=1,
    endpoint_name='churn-prediction-endpoint'
)

In [ ]:
# Make predictions via the endpoint (simulating real-time inference)

# Single prediction
sample_customer = X_test_data.iloc[0:1]
print("Single prediction request:")
print(f"  Input features: {sample_customer.values.tolist()[0]}")

response = endpoint.predict(sample_customer)
print(f"  Prediction: {'Churn' if response['predictions'][0] == 1 else 'No Churn'}")
print(f"  Probabilities: {response['probabilities'][0]}")
print()

In [ ]:
# Batch prediction via endpoint
batch_size = 10
batch_input = X_test_data.iloc[:batch_size]

print(f"Batch prediction request ({batch_size} samples):")
batch_response = endpoint.predict(batch_input)

print(f"\nPredictions vs Actual:")
print(f"{'Sample':>6} | {'Predicted':>9} | {'Actual':>6} | {'Prob(Churn)':>11} | {'Correct':>7}")
print("-" * 55)
for i in range(batch_size):
    pred = batch_response['predictions'][i]
    actual = int(y_test_actual.iloc[i])
    prob = batch_response['probabilities'][i][1]
    correct = 'Yes' if pred == actual else 'No'
    pred_label = 'Churn' if pred == 1 else 'No Churn'
    print(f"{i+1:>6} | {pred_label:>9} | {actual:>6} | {prob:>11.4f} | {correct:>7}")

In [ ]:
# Simulate JSON serialization (how real SageMaker endpoints work)
def serialize_request(data, content_type='application/json'):
    """Serialize prediction request like a real SageMaker endpoint."""
    if content_type == 'application/json':
        if isinstance(data, pd.DataFrame):
            instances = [{'features': row.tolist()} for _, row in data.iterrows()]
        elif isinstance(data, np.ndarray):
            instances = [{'features': row.tolist()} for row in data]
        else:
            instances = data
        return json.dumps({'instances': instances})
    elif content_type == 'text/csv':
        if isinstance(data, pd.DataFrame):
            return data.to_csv(index=False, header=False)
        return str(data)

def deserialize_response(response, accept='application/json'):
    """Deserialize prediction response."""
    if accept == 'application/json':
        return json.dumps(response, indent=2)
    return str(response)

# Demo serialization
sample = X_test_data.iloc[:3]
request_json = serialize_request(sample)
print("Request (JSON):")
print(request_json[:500])
print()

response = endpoint.predict(sample)
response_json = deserialize_response(response)
print("Response (JSON):")
print(response_json[:500])

In [ ]:
# Show real SageMaker deployment code
print("""
# ===== REAL SAGEMAKER DEPLOYMENT CODE =====

# Deploy to a real-time endpoint
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name='churn-prediction-v1',
    serializer=sagemaker.serializers.CSVSerializer(),
    deserializer=sagemaker.deserializers.JSONDeserializer(),
)

# Make a prediction
result = predictor.predict([[24, 79.99, 1920.50, 3, 1, 0, 1, 0, 1, 2]])
print(result)

# Auto-scaling configuration
import boto3
asg_client = boto3.client('application-autoscaling')

asg_client.register_scalable_target(
    ServiceNamespace='sagemaker',
    ResourceId=f'endpoint/churn-prediction-v1/variant/AllTraffic',
    ScalableDimension='sagemaker:variant:DesiredInstanceCount',
    MinCapacity=1,
    MaxCapacity=4,
)

asg_client.put_scaling_policy(
    PolicyName='churn-scaling-policy',
    ServiceNamespace='sagemaker',
    ResourceId=f'endpoint/churn-prediction-v1/variant/AllTraffic',
    ScalableDimension='sagemaker:variant:DesiredInstanceCount',
    PolicyType='TargetTrackingScaling',
    TargetTrackingScalingPolicyConfiguration={
        'TargetValue': 1000,  # 1000 invocations per instance
        'PredefinedMetricSpecification': {
            'PredefinedMetricType': 'SageMakerVariantInvocationsPerInstance'
        },
        'ScaleInCooldown': 300,
        'ScaleOutCooldown': 60,
    }
)

# IMPORTANT: Delete endpoint when done to stop billing!
# predictor.delete_endpoint()
""")

In [ ]:
# Clean up: delete the mock endpoint
endpoint.delete_endpoint()

---

## Section 8: SageMaker Processing Jobs

SageMaker Processing lets you run data processing, feature engineering, and
model evaluation workloads on managed infrastructure.

### Why Processing Jobs?

- **Scale**: Process terabytes of data without provisioning infrastructure
- **Reproducibility**: Containerized execution ensures consistent results
- **Integration**: Reads from/writes to S3, integrates with SageMaker Pipelines

### Common Processors

| Processor | Use Case |
|---|---|
| `SKLearnProcessor` | General preprocessing with sklearn |
| `PySparkProcessor` | Large-scale data with PySpark |
| `ScriptProcessor` | Custom container with any framework |
| `FrameworkProcessor` | TensorFlow/PyTorch data processing |

### Processing vs Lambda vs Glue

| Feature | SageMaker Processing | AWS Lambda | AWS Glue |
|---|---|---|---|
| **Max runtime** | Days | 15 minutes | Hours |
| **Max memory** | Hundreds of GB | 10 GB | Hundreds of GB |
| **ML libraries** | Full sklearn, pandas, etc. | Limited layers | PySpark |
| **GPU support** | Yes | No | No |
| **Best for** | ML preprocessing | Light transforms | ETL at scale |
| **Pricing** | Per-second instance | Per-invocation | Per-DPU-second |

In [ ]:
# Simulate a SageMaker Processing Job
# This is a data preprocessing pipeline that would run on SageMaker Processing

def simulate_processing_job(input_path, output_path, job_name='processing-job'):
    """Simulate a SageMaker Processing Job locally.
    
    In real SageMaker, this would run inside a managed container
    on a dedicated instance (e.g., ml.m5.xlarge).
    """
    print(f"\n{'='*60}")
    print(f"[SageMaker Sim] Processing Job: {job_name}")
    print(f"{'='*60}")
    print(f"Input:  {input_path}")
    print(f"Output: {output_path}")
    
    start_time = time.time()
    
    # Step 1: Load raw data
    print("\n[Processing] Step 1: Loading raw data...")
    csv_files = [f for f in os.listdir(input_path) if f.endswith('.csv')]
    raw_data = pd.read_csv(os.path.join(input_path, csv_files[0]), header=None)
    print(f"  Loaded {len(raw_data)} rows, {raw_data.shape[1]} columns")
    
    # Step 2: Data quality checks
    print("\n[Processing] Step 2: Data quality checks...")
    null_counts = raw_data.isnull().sum()
    print(f"  Null values: {null_counts.sum()} total")
    print(f"  Duplicate rows: {raw_data.duplicated().sum()}")
    
    # Step 3: Feature engineering
    print("\n[Processing] Step 3: Feature engineering...")
    processed = raw_data.copy()
    
    # Add interaction features
    processed['charges_per_month'] = processed.iloc[:, 3] / (processed.iloc[:, 1] + 1)  # total/tenure
    processed['tickets_per_month'] = processed.iloc[:, 4] / (processed.iloc[:, 1] + 1)  # tickets/tenure
    print(f"  Added 2 interaction features")
    print(f"  Final shape: {processed.shape}")
    
    # Step 4: Scale numerical features
    print("\n[Processing] Step 4: Scaling numerical features...")
    target_col = processed.iloc[:, 0]
    feature_cols = processed.iloc[:, 1:]
    
    scaler = StandardScaler()
    scaled_features = pd.DataFrame(
        scaler.fit_transform(feature_cols),
        columns=[f'scaled_{i}' for i in range(feature_cols.shape[1])]
    )
    processed_final = pd.concat([target_col.reset_index(drop=True), scaled_features], axis=1)
    print(f"  Scaled {feature_cols.shape[1]} features")
    
    # Step 5: Save processed data
    print("\n[Processing] Step 5: Saving processed data...")
    os.makedirs(output_path, exist_ok=True)
    output_file = os.path.join(output_path, 'processed.csv')
    processed_final.to_csv(output_file, index=False, header=False)
    
    # Save processing metadata
    metadata = {
        'input_rows': len(raw_data),
        'output_rows': len(processed_final),
        'input_columns': raw_data.shape[1],
        'output_columns': processed_final.shape[1],
        'null_values_found': int(null_counts.sum()),
        'duplicates_found': int(raw_data.duplicated().sum()),
        'features_added': 2,
        'processing_time_seconds': time.time() - start_time
    }
    with open(os.path.join(output_path, 'metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=2)
    
    duration = time.time() - start_time
    print(f"\n[Processing] Job complete in {duration:.2f}s")
    print(f"  Output: {output_file}")
    print(f"{'='*60}\n")
    
    return metadata


# Run the processing job
processing_output = os.path.join(DATA_DIR, 'processed')
metadata = simulate_processing_job(
    input_path=TRAIN_DIR,
    output_path=processing_output,
    job_name='churn-preprocessing'
)

print("Processing metadata:")
print(json.dumps(metadata, indent=2))

In [ ]:
# Show real SageMaker Processing code
print("""
# ===== REAL SAGEMAKER PROCESSING CODE =====

from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

sklearn_processor = SKLearnProcessor(
    framework_version='1.2-1',
    role=role,
    instance_type='ml.m5.xlarge',
    instance_count=1,
    base_job_name='churn-preprocessing',
)

sklearn_processor.run(
    code='preprocessing.py',
    inputs=[
        ProcessingInput(
            source=f's3://{bucket}/data/raw',
            destination='/opt/ml/processing/input'
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination=f's3://{bucket}/data/processed'
        )
    ]
)
""")

---

## Section 9: Cost Management

SageMaker costs can escalate quickly if not managed carefully. Understanding pricing
and using cost optimization strategies is critical for any MLOps engineer.

### Instance Type Comparison

| Instance Type | vCPUs | Memory (GB) | GPU | Price/hr (US East) | Best For |
|---|---|---|---|---|---|
| `ml.m5.large` | 2 | 8 | None | $0.115 | Small models, testing |
| `ml.m5.xlarge` | 4 | 16 | None | $0.230 | Standard ML training |
| `ml.m5.2xlarge` | 8 | 32 | None | $0.461 | Medium datasets |
| `ml.c5.xlarge` | 4 | 8 | None | $0.204 | CPU-intensive (XGBoost) |
| `ml.c5.2xlarge` | 8 | 16 | None | $0.408 | Larger CPU training |
| `ml.p3.2xlarge` | 8 | 61 | 1x V100 | $3.825 | Deep learning |
| `ml.p3.8xlarge` | 32 | 244 | 4x V100 | $14.688 | Distributed DL |
| `ml.g4dn.xlarge` | 4 | 16 | 1x T4 | $0.736 | Budget GPU training |
| `ml.g5.xlarge` | 4 | 16 | 1x A10G | $1.408 | Modern GPU training |
| `ml.inf1.xlarge` | 4 | 8 | Inferentia | $0.297 | Inference only |

### Spot Training

SageMaker Managed Spot Training uses unused EC2 capacity at up to **90% discount**:

- Training jobs automatically checkpoint and resume if interrupted
- Set `max_wait` to define the maximum time to wait for spot capacity
- Best for: training jobs that support checkpointing (most do)
- Not recommended for: very short jobs (<5 min) due to overhead

```python
# Enable spot training (real SageMaker code)
estimator = Estimator(
    ...,
    use_spot_instances=True,
    max_run=3600,        # Max training time: 1 hour
    max_wait=7200,       # Max wait for spot: 2 hours
    checkpoint_s3_uri=f's3://{bucket}/checkpoints/'
)
```

### Managed Warm Pools

For repeated training (e.g., daily retraining), warm pools keep instances provisioned
between jobs, eliminating the 3-5 minute startup time:

```python
estimator = Estimator(
    ...,
    keep_alive_period_in_seconds=3600  # Keep warm for 1 hour
)
```

In [ ]:
# Cost calculator for SageMaker training jobs

# Pricing data (US East, as of 2024)
INSTANCE_PRICING = {
    'ml.m5.large':    {'price_per_hour': 0.115, 'vcpus': 2,  'memory_gb': 8,   'gpu': None},
    'ml.m5.xlarge':   {'price_per_hour': 0.230, 'vcpus': 4,  'memory_gb': 16,  'gpu': None},
    'ml.m5.2xlarge':  {'price_per_hour': 0.461, 'vcpus': 8,  'memory_gb': 32,  'gpu': None},
    'ml.m5.4xlarge':  {'price_per_hour': 0.922, 'vcpus': 16, 'memory_gb': 64,  'gpu': None},
    'ml.c5.xlarge':   {'price_per_hour': 0.204, 'vcpus': 4,  'memory_gb': 8,   'gpu': None},
    'ml.c5.2xlarge':  {'price_per_hour': 0.408, 'vcpus': 8,  'memory_gb': 16,  'gpu': None},
    'ml.p3.2xlarge':  {'price_per_hour': 3.825, 'vcpus': 8,  'memory_gb': 61,  'gpu': '1x V100'},
    'ml.p3.8xlarge':  {'price_per_hour': 14.688,'vcpus': 32, 'memory_gb': 244, 'gpu': '4x V100'},
    'ml.g4dn.xlarge': {'price_per_hour': 0.736, 'vcpus': 4,  'memory_gb': 16,  'gpu': '1x T4'},
    'ml.g5.xlarge':   {'price_per_hour': 1.408, 'vcpus': 4,  'memory_gb': 16,  'gpu': '1x A10G'},
}


def cost_calculator(instance_type, duration_hours, instance_count=1,
                    use_spot=False, spot_discount=0.7):
    """Estimate the cost of a SageMaker training job.
    
    Args:
        instance_type: SageMaker instance type (e.g., 'ml.m5.xlarge')
        duration_hours: Expected training duration in hours
        instance_count: Number of instances (for distributed training)
        use_spot: Whether to use spot instances
        spot_discount: Expected spot discount (default 70% savings)
    
    Returns:
        dict with cost breakdown
    """
    if instance_type not in INSTANCE_PRICING:
        return {'error': f'Unknown instance type: {instance_type}'}
    
    info = INSTANCE_PRICING[instance_type]
    base_cost = info['price_per_hour'] * duration_hours * instance_count
    
    if use_spot:
        actual_cost = base_cost * (1 - spot_discount)
        savings = base_cost - actual_cost
    else:
        actual_cost = base_cost
        savings = 0
    
    return {
        'instance_type': instance_type,
        'instance_count': instance_count,
        'vcpus_total': info['vcpus'] * instance_count,
        'memory_total_gb': info['memory_gb'] * instance_count,
        'gpu': info['gpu'],
        'duration_hours': duration_hours,
        'price_per_hour': info['price_per_hour'],
        'use_spot': use_spot,
        'spot_discount': f'{spot_discount:.0%}' if use_spot else 'N/A',
        'base_cost': f'${base_cost:.2f}',
        'actual_cost': f'${actual_cost:.2f}',
        'savings': f'${savings:.2f}',
    }


# Example: Compare costs for a 2-hour training job
print("Cost Comparison: 2-hour Training Job")
print("=" * 70)

scenarios = [
    ('ml.m5.xlarge', 2, 1, False),
    ('ml.m5.xlarge', 2, 1, True),
    ('ml.c5.2xlarge', 2, 1, False),
    ('ml.c5.2xlarge', 2, 1, True),
    ('ml.p3.2xlarge', 2, 1, False),
    ('ml.p3.2xlarge', 2, 1, True),
    ('ml.g4dn.xlarge', 2, 1, False),
    ('ml.g4dn.xlarge', 2, 1, True),
]

for inst, hrs, cnt, spot in scenarios:
    result = cost_calculator(inst, hrs, cnt, spot)
    spot_label = ' (spot)' if spot else ''
    print(f"  {inst:20s}{spot_label:8s} | {result['actual_cost']:>8s} | Savings: {result['savings']:>7s}")

In [ ]:
# Visualize cost comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart: On-demand vs Spot for different instance types
instance_types = ['ml.m5.xlarge', 'ml.c5.2xlarge', 'ml.g4dn.xlarge', 'ml.p3.2xlarge']
on_demand_costs = []
spot_costs = []

for inst in instance_types:
    od = cost_calculator(inst, 2, 1, False)
    sp = cost_calculator(inst, 2, 1, True)
    on_demand_costs.append(float(od['actual_cost'].replace('$', '')))
    spot_costs.append(float(sp['actual_cost'].replace('$', '')))

x = np.arange(len(instance_types))
width = 0.35

axes[0].bar(x - width/2, on_demand_costs, width, label='On-Demand', color='#e74c3c', alpha=0.8)
axes[0].bar(x + width/2, spot_costs, width, label='Spot (70% off)', color='#2ecc71', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([t.replace('ml.', '') for t in instance_types], rotation=15)
axes[0].set_ylabel('Cost ($)')
axes[0].set_title('2-Hour Training Job: On-Demand vs Spot')
axes[0].legend()

for i, (od, sp) in enumerate(zip(on_demand_costs, spot_costs)):
    axes[0].text(i - width/2, od + 0.1, f'${od:.2f}', ha='center', fontsize=8)
    axes[0].text(i + width/2, sp + 0.1, f'${sp:.2f}', ha='center', fontsize=8)

# Line chart: Cost over time for different instance types
hours = np.arange(0.5, 25, 0.5)
for inst in ['ml.m5.xlarge', 'ml.c5.2xlarge', 'ml.g4dn.xlarge', 'ml.p3.2xlarge']:
    costs = [float(cost_calculator(inst, h)['actual_cost'].replace('$', '')) for h in hours]
    axes[1].plot(hours, costs, label=inst.replace('ml.', ''), linewidth=2)

axes[1].set_xlabel('Training Duration (hours)')
axes[1].set_ylabel('Cost ($)')
axes[1].set_title('Cumulative Cost by Instance Type')
axes[1].legend()
axes[1].axhline(y=10, color='gray', linestyle='--', alpha=0.5, label='$10 budget')
axes[1].text(24, 10.5, '$10 budget', fontsize=8, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# When NOT to use SageMaker
decision_data = {
    'Scenario': [
        'Dataset < 100 MB, simple model',
        'Dataset < 1 GB, sklearn model',
        'Dataset 1-10 GB, gradient boosting',
        'Dataset > 10 GB, distributed training',
        'Quick experiment / prototyping',
        'Deep learning with GPUs',
        'Production API with auto-scaling',
        'One-time batch scoring',
        'Automated daily retraining',
        'Hyperparameter search (20+ trials)',
    ],
    'Recommendation': [
        'Local / Jupyter',
        'Local / EC2',
        'SageMaker or EC2 (GPU)',
        'SageMaker',
        'Local',
        'SageMaker (spot)',
        'SageMaker Endpoints',
        'SageMaker Batch Transform or Lambda',
        'SageMaker Pipelines',
        'SageMaker HPO',
    ],
    'Monthly Cost Estimate': [
        '$0 (free)',
        '$0-50',
        '$10-100',
        '$50-500',
        '$0 (free)',
        '$50-500 (spot)',
        '$80-800',
        '$1-10',
        '$100-1000',
        '$20-200',
    ]
}

decision_df = pd.DataFrame(decision_data)
print("When to Use SageMaker (Decision Guide)")
print("=" * 80)
print(decision_df.to_string(index=False))

### Exercises

Put your SageMaker knowledge to the test with these exercises.

#### Exercise 1: Write a Custom Training Script for Regression

Write a SageMaker-compatible training script (`train_regression.py`) that:
1. Reads data from `SM_CHANNEL_TRAIN` and `SM_CHANNEL_VALIDATION`
2. Trains a `RandomForestRegressor` with configurable hyperparameters
3. Reports RMSE and R-squared metrics
4. Saves the model to `SM_MODEL_DIR`

In [ ]:
# Exercise 1: TODO - Write a regression training script

REGRESSION_SCRIPT = '''
# TODO: Write your SageMaker-compatible regression training script here
# Requirements:
#   - Parse hyperparameters: n_estimators, max_depth, min_samples_split
#   - Read from SM_CHANNEL_TRAIN and SM_CHANNEL_VALIDATION env vars
#   - Train RandomForestRegressor
#   - Print RMSE and R-squared for both train and validation
#   - Save model to SM_MODEL_DIR

pass
'''

# Uncomment below for the solution:

# REGRESSION_SCRIPT = '''
# import argparse
# import os
# import pickle
# import numpy as np
# import pandas as pd
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import mean_squared_error, r2_score
#
# def parse_args():
#     parser = argparse.ArgumentParser()
#     parser.add_argument('--n-estimators', type=int,
#                         default=int(os.environ.get('SM_HP_N_ESTIMATORS', 100)))
#     parser.add_argument('--max-depth', type=int,
#                         default=int(os.environ.get('SM_HP_MAX_DEPTH', 10)))
#     parser.add_argument('--min-samples-split', type=int,
#                         default=int(os.environ.get('SM_HP_MIN_SAMPLES_SPLIT', 2)))
#     parser.add_argument('--model-dir', type=str,
#                         default=os.environ.get('SM_MODEL_DIR', '/opt/ml/model'))
#     parser.add_argument('--train', type=str,
#                         default=os.environ.get('SM_CHANNEL_TRAIN', '/opt/ml/input/data/train'))
#     parser.add_argument('--validation', type=str,
#                         default=os.environ.get('SM_CHANNEL_VALIDATION', '/opt/ml/input/data/validation'))
#     return parser.parse_args()
#
# def load_data(path):
#     csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
#     df = pd.read_csv(os.path.join(path, csv_files[0]), header=None)
#     return df.iloc[:, 1:], df.iloc[:, 0]
#
# if __name__ == '__main__':
#     args = parse_args()
#     X_train, y_train = load_data(args.train)
#     model = RandomForestRegressor(
#         n_estimators=args.n_estimators,
#         max_depth=args.max_depth,
#         min_samples_split=args.min_samples_split,
#         random_state=42
#     )
#     model.fit(X_train, y_train)
#     train_pred = model.predict(X_train)
#     print(f"Train RMSE: {np.sqrt(mean_squared_error(y_train, train_pred)):.4f}")
#     print(f"Train R2: {r2_score(y_train, train_pred):.4f}")
#     if os.path.exists(args.validation):
#         X_val, y_val = load_data(args.validation)
#         val_pred = model.predict(X_val)
#         print(f"Val RMSE: {np.sqrt(mean_squared_error(y_val, val_pred)):.4f}")
#         print(f"Val R2: {r2_score(y_val, val_pred):.4f}")
#     os.makedirs(args.model_dir, exist_ok=True)
#     with open(os.path.join(args.model_dir, 'model.pkl'), 'wb') as f:
#         pickle.dump(model, f)
#     print("Model saved.")
# '''

print("Exercise 1: Write your regression training script above.")

#### Exercise 2: Configure HPO for Classification with 3+ Hyperparameters

Using the `SageMakerSimulator.HyperparameterTuner`, configure an HPO job that:
1. Tunes at least 4 hyperparameters for a Random Forest classifier
2. Uses at least 12 trials
3. Maximizes validation F1 score
4. Visualizes the results

In [ ]:
# Exercise 2: TODO - Configure HPO for classification

# TODO: Define hyperparameter ranges for Random Forest
# rf_hp_ranges = {
#     'n_estimators': ...,
#     'max_depth': ...,
#     'min_samples_split': ...,
#     ... at least one more ...
# }

# TODO: Create base estimator with algorithm='random_forest'

# TODO: Create HyperparameterTuner with max_jobs=12, objective='f1'

# TODO: Run tuner.fit() and display results

print("Exercise 2: Configure and run HPO above.")

# Uncomment below for the solution:

# rf_hp_ranges = {
#     'n_estimators': (50, 300),
#     'max_depth': (3, 15),
#     'min_samples_split': (2, 20),
#     'subsample': (0.6, 1.0),  # Note: RF doesn't use subsample, but simulator handles it
# }
#
# rf_base = SageMakerSimulator.Estimator(
#     algorithm='random_forest',
#     instance_type='ml.m5.xlarge',
#     output_path=MODEL_DIR,
#     base_job_name='churn-rf-hpo'
# )
#
# rf_tuner = SageMakerSimulator.HyperparameterTuner(
#     estimator=rf_base,
#     hyperparameter_ranges=rf_hp_ranges,
#     objective_metric_name='f1',
#     objective_type='Maximize',
#     max_jobs=12,
#     max_parallel_jobs=3,
#     strategy='Bayesian'
# )
#
# rf_tuner.fit({'train': TRAIN_DIR, 'validation': VAL_DIR})
#
# rf_results = rf_tuner.results_as_dataframe().sort_values('score', ascending=False)
# print(rf_results.to_string(index=False, float_format='%.4f'))
# print(f"\nBest F1: {rf_tuner.best_score_:.4f}")
# print(f"Best params: {rf_tuner.best_params_}")

#### Exercise 3: Design an Auto-Scaling Policy for an Endpoint

Given the following requirements, design an auto-scaling configuration:
- Peak traffic: 5000 requests/minute
- Off-peak traffic: 100 requests/minute
- Latency SLA: p99 < 200ms
- Budget: $500/month maximum

Calculate the instance type, min/max instances, and scaling policy.

In [ ]:
# Exercise 3: TODO - Design auto-scaling policy

# TODO: Calculate how many instances needed for peak traffic
# Hint: ml.m5.xlarge can handle ~500 requests/min for a typical sklearn model

# TODO: Calculate monthly cost for min and max instances

# TODO: Define the scaling policy (target metric, thresholds, cooldown)

print("Exercise 3: Design your auto-scaling policy above.")

# Uncomment below for the solution:

# # Capacity planning
# requests_per_min_per_instance = 500  # Typical for sklearn on ml.m5.xlarge
# peak_requests = 5000
# off_peak_requests = 100
#
# max_instances = int(np.ceil(peak_requests / requests_per_min_per_instance))  # 10
# min_instances = max(1, int(np.ceil(off_peak_requests / requests_per_min_per_instance)))  # 1
#
# # Cost calculation
# price_per_hour = INSTANCE_PRICING['ml.m5.xlarge']['price_per_hour']  # $0.23
# hours_per_month = 730
#
# # Assume: 8 hours peak, 16 hours off-peak per day
# peak_hours = 8 * 30  # 240 hours
# off_peak_hours = 16 * 30  # 480 hours
#
# avg_peak_instances = max_instances * 0.7  # ~70% utilization during peak
# monthly_cost = (peak_hours * avg_peak_instances * price_per_hour +
#                 off_peak_hours * min_instances * price_per_hour)
#
# print(f"Instance type: ml.m5.xlarge")
# print(f"Min instances: {min_instances}")
# print(f"Max instances: {max_instances}")
# print(f"Estimated monthly cost: ${monthly_cost:.2f}")
# print(f"Budget: $500/month {'(within budget)' if monthly_cost <= 500 else '(OVER BUDGET!)'}")
# print(f"\nScaling policy:")
# print(f"  Metric: SageMakerVariantInvocationsPerInstance")
# print(f"  Target: {requests_per_min_per_instance} invocations/instance/minute")
# print(f"  Scale-out cooldown: 60 seconds")
# print(f"  Scale-in cooldown: 300 seconds")

---

## Section 10: Key Takeaways

### Core Lessons

1. **SageMaker is for scale, local is for prototyping.** Always start local. Only move to
   SageMaker when you need managed compute, distributed training, or production endpoints.

2. **Script Mode is the sweet spot** between built-in algorithms and fully custom containers.
   It lets you use any Python code while SageMaker handles infrastructure. This is how
   most real-world SageMaker training works.

3. **Always calculate cost before launching training jobs.** A forgotten `ml.p3.8xlarge`
   endpoint running for a weekend costs ~$5,300. Use the cost calculator to estimate
   before you launch.

4. **HPO should be budget-constrained, not exhaustive.** Set `max_jobs` based on your budget,
   not the size of the search space. Bayesian optimization finds good solutions in 10-20 trials.

5. **Test your training script locally before cloud deployment.** Set `SM_CHANNEL_TRAIN`,
   `SM_MODEL_DIR`, etc. as environment variables and run your script with `python train.py`.
   This catches 90% of bugs for free.

6. **Spot instances are almost always worth using** for training jobs. The 60-90% savings
   far outweigh the occasional interruption (which SageMaker handles with checkpoints).

7. **Delete endpoints when you are done!** Unlike training jobs, endpoints bill continuously
   while deployed. Set up billing alerts in AWS to catch forgotten endpoints.

### SageMaker Decision Flowchart

```
Is your data > 10 GB?  -----Yes-----> SageMaker Training
       |
       No
       |
Do you need GPUs?  ---------Yes-----> SageMaker Training (spot)
       |
       No
       |
Is this for production? ----Yes-----> SageMaker Endpoints
       |
       No
       |
Train locally. Deploy later when ready.
```

### What's Next

In **Notebook 03: Docker and FastAPI**, we will:
- Containerize ML models with Docker
- Build prediction APIs with FastAPI
- Create SageMaker-compatible custom containers
- Understand how SageMaker uses Docker under the hood